In [ ]:
# Run this in a notebook cell if libraries are missing
#!pip install pandas scikit-learn tensorflow matplotlib seaborn

   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/8.7 MB 3.2 MB/s eta 0:00:03
   --------- ------------------------------ 2.1/8.7 MB 4.0 MB/s eta 0:00:02
   -------------- ------------------------- 3.1/8.7 MB 4.4 MB/s eta 0:00:02
   ------------------ --------------------- 3.9/8.7 MB 4.5 MB/s eta 0:00:02
   ------------------------ --------------- 5.2/8.7 MB 4.5 MB/s eta 0:00:01
   --------------------------- ------------ 6.0/8.7 MB 4.3 MB/s eta 0:00:01
   ------------------------------- -------- 6.8/8.7 MB 4.3 MB/s eta 0:00:01
   ------------------------------------ --- 7.9/8.7 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------  8.7/8.7 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 4.2 MB/s  0:00:02
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   -------------------------------

# Block 1: Imports and Setup

In [9]:


import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
INPUT_CSV_PATH = 'stockprice.csv'  # <-- Update this path if needed

# Identify the directory of the input file to create output folders there
BASE_DIR = os.path.dirname(os.path.abspath(INPUT_CSV_PATH)) if os.path.exists(INPUT_CSV_PATH) else '.'

# Create output directories
OUTPUT_FIGURES_DIR = os.path.join(BASE_DIR, 'output_figures')
OUTPUT_MODELS_DIR = os.path.join(BASE_DIR, 'trained_models')
OUTPUT_RESULTS_DIR = os.path.join(BASE_DIR, 'results')

os.makedirs(OUTPUT_FIGURES_DIR, exist_ok=True)
os.makedirs(OUTPUT_MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

print(f"Input CSV: {INPUT_CSV_PATH}")
print(f"Figures will be saved to: {OUTPUT_FIGURES_DIR}")
print(f"Models will be saved to: {OUTPUT_MODELS_DIR}")
print(f"Results will be saved to: {OUTPUT_RESULTS_DIR}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

Input CSV: stockprice.csv
Figures will be saved to: d:\AML Project\output_figures
Models will be saved to: d:\AML Project\trained_models
Results will be saved to: d:\AML Project\results


# Block 2: Load and Inspect Data

In [10]:
# Load the dataset
try:
    df = pd.read_csv(INPUT_CSV_PATH)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: File {INPUT_CSV_PATH} not found.")
    raise

# Display basic info
print("\n--- Dataset Info ---")
print(df.info())

print("\n--- First few rows ---")
print(df.head())

print("\n--- Unique Sectors ---")
sectors = df['sector'].unique()
print(sectors)

# Check for any potential date parsing issues or missing values
print("\n--- Checking for missing values ---")
print(df.isnull().sum())

# Ensure 'trading date' is datetime
df['trading_date'] = pd.to_datetime(df['trading_date'])

# Sort data by sector and date
df = df.sort_values(['sector', 'trading_date']).reset_index(drop=True)
print("\nData sorted by sector and date.")

Data loaded successfully.

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4270 entries, 0 to 4269
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sector        4270 non-null   object 
 1   trading_date  4270 non-null   object 
 2   open          4270 non-null   float64
 3   high          4270 non-null   float64
 4   low           4270 non-null   float64
 5   close         4270 non-null   float64
 6   volume        4269 non-null   float64
 7   year          4270 non-null   int64  
 8   month         4270 non-null   int64  
 9   date          4270 non-null   int64  
dtypes: float64(5), int64(3), object(2)
memory usage: 333.7+ KB
None

--- First few rows ---
        sector trading_date    open    high     low   close     volume  year  \
0  Engineering   2017-05-02  101.57  102.57   99.90  101.13  399156.61  2017   
1  Engineering   2017-05-03  101.41  103.05  100.22  101.16  328356.39  2017

# Block 3: Define time2vec Layer

In [11]:

class Time2Vec(layers.Layer):
    def __init__(self, kernel_size=1, **kwargs):
        super(Time2Vec, self).__init__(**kwargs)
        self.k = kernel_size

    def build(self, input_shape):
        # For simplicity, we'll learn a linear and a periodic component for the last feature (e.g., time proxy)
        # In a full implementation, you might use actual timestamps
        self.w_linear = self.add_weight(shape=(input_shape[-1], 1), initializer="uniform", trainable=True, name='w_linear')
        self.b_linear = self.add_weight(shape=(1,), initializer="uniform", trainable=True, name='b_linear')
        
        self.w_periodic = self.add_weight(shape=(input_shape[-1], self.k), initializer="uniform", trainable=True, name='w_periodic')
        self.b_periodic = self.add_weight(shape=(self.k,), initializer="uniform", trainable=True, name='b_periodic')
        super(Time2Vec, self).build(input_shape)

    def call(self, inputs):
        # Linear part: (batch, time, features) * (features, 1) -> (batch, time, 1)
        linear = tf.matmul(inputs, self.w_linear) + self.b_linear
        
        # Periodic part: sin((batch, time, features) * (features, k) + b) -> (batch, time, k)
        periodic = tf.math.sin(tf.matmul(inputs, self.w_periodic) + self.b_periodic)
        
        # Concatenate: (batch, time, 1) + (batch, time, k) -> (batch, time, k+1)
        return tf.concat([linear, periodic], axis=-1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], self.k + 1)

# Test the layer
# test_input = tf.random.normal((2, 8, 5))
# t2v_layer = Time2Vec(kernel_size=1)
# print(t2v_layer(test_input).shape) # Should be (2, 8, 2)

# Block 4: Data Preprocessing

In [ ]:
'''
def preprocess_sector_data(group):
    """
    Preprocesses data for a single sector group.
    Applies 10-day MA smoothing, differencing, and normalization.
    """
    group = group.copy()
    cols = ['open', 'high', 'low', 'close', 'volume']
    
    # 1. Apply 10-day moving average smoothing
    smoothed = group[cols].rolling(window=10, min_periods=1).mean()
    
    # 2. Compute returns (differencing) on smoothed data
    for col in cols:
        group[f'return_{col}'] = smoothed[col].diff()
    
    # 3. Drop rows with NaN (from diff and rolling)
    group = group.dropna().reset_index(drop=True)
    
    return group

def create_sequences(data, seq_length=8):
    """
    Creates input sequences and corresponding labels.
    """
    X, y, last_closes = [], [], []
    feature_cols = [f'return_{f}' for f in ['open', 'high', 'low', 'close', 'volume']]
    
    for i in range(seq_length, len(data)):
        seq = data[feature_cols].iloc[i-seq_length:i].values
        label = data['return_close'].iloc[i]
        last_close = data['close'].iloc[i-1] # Close price of the day before prediction
        
        X.append(seq)
        y.append(label)
        last_closes.append(last_close)
        
    return np.array(X), np.array(y), np.array(last_closes)

# Dictionary to hold scalers for each sector (for inverse transform later)
scalers = {}
processed_data_dict = {}

# Process data for each sector
for sector in sectors:
    print(f"Preprocessing data for sector: {sector}")
    sector_data = df[df['sector'] == sector].copy()
    
    # Apply preprocessing
    processed_data = preprocess_sector_data(sector_data)
    
    # Normalize features
    feature_cols = [f'return_{f}' for f in ['open', 'high', 'low', 'close', 'volume']]
    scaler = MinMaxScaler()
    processed_data[feature_cols] = scaler.fit_transform(processed_data[feature_cols])
    scalers[sector] = scaler # Store scaler
    
    # Create sequences
    X, y, last_closes = create_sequences(processed_data, seq_length=8)
    processed_data_dict[sector] = {
        'X': X, 'y': y, 'last_closes': last_closes, 'raw_data': processed_data
    }
    print(f"  -> Sequences created: {X.shape}, Labels: {y.shape}")

print("Preprocessing complete for all sectors.")
'''

Preprocessing data for sector: Engineering
  -> Sequences created: (845, 8, 5), Labels: (845,)
Preprocessing data for sector: Fuel & Power
  -> Sequences created: (845, 8, 5), Labels: (845,)
Preprocessing data for sector: IT Sector
  -> Sequences created: (844, 8, 5), Labels: (844,)
Preprocessing data for sector: Services & Real Estate
  -> Sequences created: (845, 8, 5), Labels: (845,)
Preprocessing data for sector: Telecommunication
  -> Sequences created: (845, 8, 5), Labels: (845,)
Preprocessing complete for all sectors.


In [12]:
# Block 4: Data Preprocessing Function (Corrected Label Shape)

def preprocess_sector_data(group):
    """
    Preprocesses data for a single sector group.
    Applies 10-day MA smoothing, differencing, and normalization.
    """
    group = group.copy()
    cols = ['open', 'high', 'low', 'close', 'volume']
    
    # 1. Apply 10-day moving average smoothing
    smoothed = group[cols].rolling(window=10, min_periods=1).mean()
    
    # 2. Compute returns (differencing) on smoothed data
    for col in cols:
        group[f'return_{col}'] = smoothed[col].diff()
    
    # 3. Drop rows with NaN (from diff and rolling)
    group = group.dropna().reset_index(drop=True)
    return group

def create_sequences(data, seq_length=8):
    """
    Creates input sequences and corresponding labels.
    Ensures labels `y` have the correct shape (samples, 1).
    """
    X, y, last_closes = [], [], []
    feature_cols = [f'return_{f}' for f in ['open', 'high', 'low', 'close', 'volume']]
    
    for i in range(seq_length, len(data)):
        seq = data[feature_cols].iloc[i-seq_length:i].values
        # --- Key Fix: Reshape label to be (1,) ---
        # This ensures y is a column vector (samples, 1) after collection
        label = data['return_close'].iloc[i] 
        last_close = data['close'].iloc[i-1] # Close price of the day before prediction
        
        X.append(seq)
        y.append(label)
        last_closes.append(last_close)
        
    # --- Key Fix: Convert lists to numpy arrays and ensure y has shape (samples, 1) ---
    X = np.array(X)
    # Reshape y to be a 2D array with one column
    y = np.array(y).reshape(-1, 1) 
    last_closes = np.array(last_closes)
        
    return X, y, last_closes # X: (samples, 8, 5), y: (samples, 1), last_closes: (samples,)

# Dictionary to hold scalers for each sector (for inverse transform later)
scalers = {}
processed_data_dict = {}

# Process data for each sector
for sector in sectors:
    print(f"Preprocessing data for sector: {sector}")
    sector_data = df[df['sector'] == sector].copy()
    
    # Apply preprocessing
    processed_data = preprocess_sector_data(sector_data)
    
    # Normalize features
    feature_cols = [f'return_{f}' for f in ['open', 'high', 'low', 'close', 'volume']]
    scaler = MinMaxScaler()
    processed_data[feature_cols] = scaler.fit_transform(processed_data[feature_cols])
    scalers[sector] = scaler # Store scaler
    
    # Create sequences
    X, y, last_closes = create_sequences(processed_data, seq_length=8)
    processed_data_dict[sector] = {
        'X': X, 'y': y, 'last_closes': last_closes, 'raw_data': processed_data
    }
    print(f"  -> Sequences created: X={X.shape}, y={y.shape}, Labels: {y.shape}")

print("Preprocessing complete for all sectors.")

Preprocessing data for sector: Engineering
  -> Sequences created: X=(845, 8, 5), y=(845, 1), Labels: (845, 1)
Preprocessing data for sector: Fuel & Power
  -> Sequences created: X=(845, 8, 5), y=(845, 1), Labels: (845, 1)
Preprocessing data for sector: IT Sector
  -> Sequences created: X=(844, 8, 5), y=(844, 1), Labels: (844, 1)
Preprocessing data for sector: Services & Real Estate
  -> Sequences created: X=(845, 8, 5), y=(845, 1), Labels: (845, 1)
Preprocessing data for sector: Telecommunication
  -> Sequences created: X=(845, 8, 5), y=(845, 1), Labels: (845, 1)
Preprocessing complete for all sectors.


# Block 5: Define the Transformer Model

In [13]:
def build_transformer_model(seq_length=8, n_features=5, d_model=32, n_heads=2, d_ff=64, dropout=0.1):
    """
    Builds the Transformer model with time2vec encoding.
    """
    inputs = layers.Input(shape=(seq_length, n_features), name='sequence_input')
    
    # --- time2vec Encoding ---
    # Apply Time2Vec to each timestep. This is a simplified version.
    # A more robust version would use actual time features.
    time_embeddings = Time2Vec(kernel_size=1)(inputs) # Output: (batch, 8, 2)
    
    # Project original features to d_model
    linear_proj = layers.Dense(d_model, name='feature_projection')(inputs) # (batch, 8, d_model)
    
    # Project time2vec embeddings to d_model for addition
    time_proj = layers.Dense(d_model, name='time_projection')(time_embeddings) # (batch, 8, d_model)
    
    # Combine features and time embeddings
    x = layers.Add(name='feature_time_fusion')([linear_proj, time_proj])
    
    # --- Positional Encoding (Simple Embedding) ---
    positions = tf.range(start=0, limit=seq_length, delta=1)
    pos_encoding_layer = layers.Embedding(input_dim=seq_length, output_dim=d_model, name='positional_encoding')
    pos_encodings = pos_encoding_layer(positions) # (8, d_model)
    x = layers.Add()([x, pos_encodings]) # Broadcast add (batch, 8, d_model)
    
    # --- Transformer Encoder Layer ---
    # Multi-Head Attention
    attn_output = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model, name='multi_head_attention')(x, x)
    attn_output = layers.Dropout(dropout)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6, name='ln_after_attention')(attn_output + x) # Residual connection
    
    # Feed Forward
    ffn_output = layers.Dense(d_ff, activation='relu', name='ffn_dense1')(out1)
    ffn_output = layers.Dense(d_model, name='ffn_dense2')(ffn_output)
    ffn_output = layers.Dropout(dropout)(ffn_output)
    out2 = layers.LayerNormalization(epsilon=1e-6, name='ln_after_ffn')(ffn_output + out1) # Residual connection
    
    # --- Global Pooling and Final Layers ---
    pooled = layers.GlobalAveragePooling1D(name='global_avg_pool')(out2) # (batch, d_model)
    dense1 = layers.Dense(32, activation='relu', name='final_dense1')(pooled)
    dropout_final = layers.Dropout(0.1, name='final_dropout')(dense1)
    outputs = layers.Dense(1, name='output_layer')(dropout_final) # Predict single return value
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Transformer_Stock_Predictor")
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Test model creation
# test_model = build_transformer_model()
# test_model.summary()

# Block 6: Train Models and Evaluate

In [14]:
# Dictionary to store results
results = {}
model_histories = {}

# Hyperparameters
EPOCHS = 50
BATCH_SIZE = 32

for sector in sectors:
    print(f"\n{'='*20} Training Model for {sector} {'='*20}")
    
    # 1. Get processed data
    data_dict = processed_data_dict[sector]
    X, y, last_closes = data_dict['X'], data_dict['y'], data_dict['last_closes']
    scaler = scalers[sector]
    
    if len(X) == 0:
        print(f"Skipping {sector} due to insufficient data after preprocessing.")
        continue

    # 2. Split data (80:10:10)
    n_total = len(X)
    n_train = int(0.8 * n_total)
    n_val = int(0.1 * n_total)
    # n_test = n_total - n_train - n_val

    X_train, y_train = X[:n_train], y[:n_train]
    X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
    X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]
    last_closes_test = last_closes[n_train+n_val:]
    
    print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

    # 3. Build and Train Model
    model = build_transformer_model()
    # model.summary() # Optional: print model summary

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1 # Set to 0 for less output
    )
    
    model_histories[sector] = history
    
    # 4. Evaluate on Test Set
    y_pred_norm = model.predict(X_test, verbose=0).flatten()
    
    # 5. Calculate Metrics on Normalized Data
    rmse_norm = np.sqrt(mean_squared_error(y_test, y_pred_norm))
    mae_norm = mean_absolute_error(y_test, y_pred_norm)
    
    # 6. Inverse Transform Predictions and True Labels to get actual returns
    # We need to inverse transform the target (close return)
    # The scaler was fit on all 5 return features, so we need to create a dummy array
    # to inverse transform just the close return.
    # A cleaner way is to have a separate scaler for the target.
    # For simplicity here, we assume the scaler can handle inverse on a single column
    # by reshaping. This works if the scaler was fit on the target column last or we know its index.
    # Let's assume 'return_close' was the 4th feature (index 3) during scaling.
    # We create a dummy array with zeros for other features and the predicted norm value for close return.
    
    # More robust approach: Inverse transform using the scaler's parameters directly
    y_test_actual_returns = scaler.inverse_transform(
        np.hstack([
            np.zeros((len(y_test), 4)), # Dummy values for open, high, low, volume returns
            y_test.reshape(-1, 1)       # Actual close return to inverse transform
        ])
    )[:, 4] # Get the 5th column (index 4) which is the inverted close return

    y_pred_actual_returns = scaler.inverse_transform(
        np.hstack([
            np.zeros((len(y_pred_norm), 4)),
            y_pred_norm.reshape(-1, 1)
        ])
    )[:, 4]

    # 7. Calculate Final Metrics on Actual Returns
    rmse_actual = np.sqrt(mean_squared_error(y_test_actual_returns, y_pred_actual_returns))
    mae_actual = mean_absolute_error(y_test_actual_returns, y_pred_actual_returns)
    
    # 8. Reconstruct Actual Closing Prices
    # Predicted Close = Last Known Close + Predicted Return
    predicted_closing_prices = last_closes_test + y_pred_actual_returns
    actual_closing_prices = last_closes_test + y_test_actual_returns # This should be the same as the 'close' column in raw data for these test dates
    
    # Store results
    results[sector] = {
        'RMSE_Norm': rmse_norm, 'MAE_Norm': mae_norm,
        'RMSE_Actual_Return': rmse_actual, 'MAE_Actual_Return': mae_actual,
        'y_test_actual_returns': y_test_actual_returns,
        'y_pred_actual_returns': y_pred_actual_returns,
        'actual_closing_prices': actual_closing_prices,
        'predicted_closing_prices': predicted_closing_prices,
        'test_dates_indices': np.arange(n_train+n_val, n_total) # To align with raw data if needed
    }
    
    # 9. Save Model
    model_path = os.path.join(OUTPUT_MODELS_DIR, f"model_{sector.replace(' ', '_')}.keras")
    model.save(model_path)
    print(f"  Model saved to: {model_path}")
    
    print(f"  Test RMSE (Actual Returns): {rmse_actual:.6f}")
    print(f"  Test MAE (Actual Returns): {mae_actual:.6f}")

print("\nTraining and evaluation completed for all sectors.")


==================== Training Model for Engineering ====================
  Train: (676, 8, 5), Val: (84, 8, 5), Test: (85, 8, 5)
Epoch 1/50


InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/compile_loss/mse/sub/BroadcastGradientArgs defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "C:\Users\user\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\user\AppData\Roaming\Python\Python313\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "C:\Users\user\AppData\Roaming\Python\Python313\site-packages\ipykernel\kernelapp.py", line 739, in start

  File "C:\Users\user\AppData\Roaming\Python\Python313\site-packages\tornado\platform\asyncio.py", line 205, in start

  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\asyncio\base_events.py", line 679, in run_forever

  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\asyncio\base_events.py", line 2027, in _run_once

  File "c:\Users\user\

# Block 7: Generate Results Table

In [ ]:
# Create a summary DataFrame
results_summary_data = []
for sector, metrics in results.items():
    results_summary_data.append({
        'Sector': sector,
        'Daily_RMSE': metrics['RMSE_Actual_Return'],
        'Daily_MAE': metrics['MAE_Actual_Return']
    })

results_df = pd.DataFrame(results_summary_data)
# Sort by Sector name for consistent order
results_df = results_df.sort_values('Sector').reset_index(drop=True)

# Save to CSV
results_csv_path = os.path.join(OUTPUT_RESULTS_DIR, "model_performance_summary.csv")
results_df.to_csv(results_csv_path, index=False)
print(f"Results summary saved to: {results_csv_path}")
print(results_df)

# Block 8: Generate Graphs

In [ ]:
def plot_training_history(history, sector_name, save_path):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.history['loss'], label='Training Loss')
    ax.plot(history.history['val_loss'], label='Validation Loss')
    ax.set_title(f'Model Loss for {sector_name}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

def plot_price_prediction(actual_prices, predicted_prices, sector_name, save_path, num_points=100):
    # Plot last N points for clarity
    if len(actual_prices) > num_points:
        actual_prices = actual_prices[-num_points:]
        predicted_prices = predicted_prices[-num_points:]
        
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(actual_prices, label='Actual Closing Price', marker='o', markersize=3)
    ax.plot(predicted_prices, label='Predicted Closing Price', marker='x', markersize=3)
    ax.set_title(f'Closing Price Prediction vs Actual - {sector_name} (Last {len(actual_prices)} Days)')
    ax.set_xlabel('Time (Test Set Index)')
    ax.set_ylabel('Closing Price')
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)

# Generate plots for each sector
for sector in sectors:
    if sector not in results:
        continue # Skip if model wasn't trained/evaluated

    # 1. Training History Plot
    history = model_histories[sector]
    history_plot_path = os.path.join(OUTPUT_FIGURES_DIR, f"history_{sector.replace(' ', '_')}.png")
    plot_training_history(history, sector, history_plot_path)
    print(f"Saved training history plot for {sector}")

    # 2. Prediction vs Actual Price Plot
    res = results[sector]
    price_plot_path = os.path.join(OUTPUT_FIGURES_DIR, f"prediction_{sector.replace(' ', '_')}.png")
    plot_price_prediction(res['actual_closing_prices'], res['predicted_closing_prices'], sector, price_plot_path)
    print(f"Saved prediction plot for {sector}")

print("All graphs generated and saved.")

# Block 9: (Optional) Weekly Model Extension

In [ ]:
# This block is a placeholder to indicate where weekly model logic would go.
# It would involve:
# 1. Resampling daily data to weekly (Open, High, Low, Close, Volume).
# 2. Applying the same preprocessing pipeline (MA smoothing, differencing, normalization).
# 3. Creating sequences.
# 4. Training a separate model for each sector.
# 5. Evaluating and saving results/plots similarly.
#
# Due to length, this is not implemented here but follows the same structure.
# You can uncomment and implement this section if needed.

print("\n--- Starting Weekly Model Extension ---")

# Function to resample to weekly
def resample_to_weekly(group):
    # Assuming DSE week is Sun-Thu
    group = group.set_index('trading date')
    weekly_data = group.resample('W-THU').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna()
    weekly_data = weekly_data.reset_index()
    return weekly_data

# Process weekly data for each sector
weekly_data_dict = {}
for sector in sectors:
    print(f"Resampling data for sector: {sector} (Weekly)")
    sector_data = df[df['sector'] == sector].copy()
    weekly_data = resample_to_weekly(sector_data)
    # Apply same preprocessing steps as daily (preprocess_sector_data)
    # ... (implement preprocessing for weekly data)
    # weekly_data_dict[sector] = processed_weekly_data

# Then, train models similarly...
# This part is left as an exercise to keep the core daily model implementation clear.